# Face 03 · Embedding Training (IResNet-50 teacher + AdaFace)

**Architecture (most robust for phone selfies in garages):**
- Server / teacher: **IResNet-50** backbone → 512-d embedding (BN-Dropout-FC-BN).
- Loss: **AdaFace** (margin adapts to image quality via feature norm) – clearly better than ArcFace on
  low-light, blurry, low-resolution faces, which is exactly the check-in condition.
- Input 112×112 RGB aligned, normalised to [-1, 1]. Mixed precision, SGD + cosine, 24–26 epochs on WebFace4M.

Notebook 04 distils this teacher into the on-device **MobileFaceNet** (< 5 MB).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
device = "cuda" if torch.cuda.is_available() else "cpu"

class IBasicBlock(nn.Module):
    def __init__(self, inp, out, stride):
        super().__init__()
        self.body = nn.Sequential(nn.BatchNorm2d(inp), nn.Conv2d(inp, out, 3, 1, 1, bias=False), nn.BatchNorm2d(out), nn.PReLU(out),
                                  nn.Conv2d(out, out, 3, stride, 1, bias=False), nn.BatchNorm2d(out))
        self.short = None if stride == 1 and inp == out else nn.Sequential(nn.Conv2d(inp, out, 1, stride, bias=False), nn.BatchNorm2d(out))
    def forward(self, x):
        return self.body(x) + (x if self.short is None else self.short(x))

class IResNet(nn.Module):
    def __init__(self, layers=(3, 4, 14, 3), dim=512):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.PReLU(64))
        blocks, inp = [], 64
        for out, n in zip((64, 128, 256, 512), layers):
            for i in range(n):
                blocks.append(IBasicBlock(inp, out, 2 if i == 0 else 1)); inp = out
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.BatchNorm2d(512), nn.Dropout(0.2), nn.Flatten(), nn.Linear(512 * 7 * 7, dim), nn.BatchNorm1d(dim))
    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))

print(sum(p.numel() for p in IResNet().parameters()) / 1e6, "M params")

In [ ]:
class AdaFace(nn.Module):
    """AdaFace head (Kim et al., CVPR 2022): margin scaled by normalised feature norm (image quality proxy)."""
    def __init__(self, dim, classes, m=0.4, h=0.333, s=64.0, t_alpha=0.01):
        super().__init__()
        self.W = nn.Parameter(torch.randn(dim, classes) * 0.01)
        self.m, self.h, self.s, self.t_alpha = m, h, s, t_alpha
        self.register_buffer("batch_mean", torch.ones(1) * 20)
        self.register_buffer("batch_std", torch.ones(1) * 100)

    def forward(self, emb, labels):
        norms = emb.norm(dim=1, keepdim=True).clamp(0.001, 100)
        cos = (F.normalize(emb) @ F.normalize(self.W, dim=0)).clamp(-1 + 1e-3, 1 - 1e-3)
        with torch.no_grad():
            self.batch_mean = (1 - self.t_alpha) * self.batch_mean + self.t_alpha * norms.mean()
            self.batch_std = (1 - self.t_alpha) * self.batch_std + self.t_alpha * norms.std()
        margin_scaler = ((norms - self.batch_mean) / (self.batch_std + 1e-3) * self.h).clamp(-1, 1)
        one_hot = F.one_hot(labels, cos.size(1)).bool()
        theta = cos.acos()
        g_ang = -self.m * margin_scaler          # angular margin
        g_add = self.m + self.m * margin_scaler  # additive margin
        theta_m = torch.where(one_hot, (theta + g_ang).clamp(1e-3, math.pi - 1e-3), theta)
        logits = torch.where(one_hot, theta_m.cos() - g_add, cos)
        return logits * self.s

In [ ]:
tf = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.3, 0.3, 0.2),
                         transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
                         transforms.ToTensor(), transforms.Normalize([0.5] * 3, [0.5] * 3)])
train = datasets.ImageFolder(DATA / "Faces" / "Train", transform=tf)
loader = DataLoader(train, batch_size=256, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)

backbone = IResNet().to(device)
head = AdaFace(512, len(train.classes)).to(device)
params = [{"params": backbone.parameters(), "weight_decay": 5e-4}, {"params": head.parameters(), "weight_decay": 5e-4}]
optim = torch.optim.SGD(params, lr=0.1, momentum=0.9)
EPOCHS = 26
sched = torch.optim.lr_scheduler.MultiStepLR(optim, milestones=[12, 20, 24], gamma=0.1)
scaler = torch.amp.GradScaler(enabled=device == "cuda")

In [ ]:
for epoch in range(EPOCHS):
    backbone.train(); head.train()
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device)
        with torch.autocast(device, enabled=device == "cuda"):
            loss = F.cross_entropy(head(backbone(images).float(), labels), labels)
        optim.zero_grad(); scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
    sched.step()
    torch.save(backbone.state_dict(), RUNS / "iresnet50_adaface.pt")
    print(f"epoch {epoch} loss {loss.item():.3f}")